# 06 - Data Assessment

## Purpose
Load all processed datasets and describe them. Shape, dtypes, nulls, value
ranges, country name formats, CN code formats, and cross-dataset consistency.
No cleaning or transformation. Observation and notes only.

## Inputs
- `data/processed/cbam_defaults.csv`
- `data/processed/eu_import_trade_flows.csv`
- `data/processed/country_grid_electricity.csv`
- `data/processed/hydrogen_route_intensities.csv`
- `data/processed/steel_route_intensity.csv`

## Notes
- Findings from this notebook feed directly into `07_clean_and_align.ipynb`.
- Each section covers one dataset, followed by a cross-dataset section
  focused on join readiness.

In [13]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

processed = Path("../data/processed")

defaults  = pd.read_csv(processed / "cbam_defaults.csv")
flows     = pd.read_csv(processed / "eu_import_trade_flows.csv")
grid      = pd.read_csv(processed / "country_grid_electricity.csv")
hydrogen  = pd.read_csv(processed / "hydrogen_route_intensities.csv")
steel     = pd.read_csv(processed / "steel_route_intensity.csv")

datasets = {
    "cbam_defaults":              defaults,
    "eu_import_trade_flows":      flows,
    "country_grid_electricity":   grid,
    "iron_steel_route_mapping":   route_map,
    "hydrogen_route_intensities": hydrogen,
    "steel_route_intensity":      steel,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

cbam_defaults: (10671, 10)
eu_import_trade_flows: (165182, 9)
country_grid_electricity: (193936, 10)
iron_steel_route_mapping: (335, 4)
hydrogen_route_intensities: (6, 5)
steel_route_intensity: (12, 4)


## 1. CBAM Defaults 

(`cbam_defaults.csv`)

119 countries, all CBAM-covered CN codes. One row per country/CN code/description
combination. Emission values in tCO2 per tonne of product.

In [3]:
# Basic structure check
print("Shape:", defaults.shape)
print("\nDtypes:")
print(defaults.dtypes)
print("\nSample:")
defaults.head(3)

Shape: (10671, 10)

Dtypes:
country                     str
cn_code                     str
description                 str
direct_emissions        float64
indirect_emissions      float64
total_emissions         float64
default_2026            float64
default_2027            float64
default_2028_onwards    float64
production_route            str
dtype: object

Sample:


,country,cn_code,description,direct_emissions,indirect_emissions,total_emissions,default_2026,default_2027,default_2028_onwards,production_route
0,Albania,2523 10 00,Grey clinker,0.8700,0.0000,0.8700,0.9570,1.0440,1.1310,(A)
1,Albania,2523 29 00,Grey Portland cement,0.9000,0.0300,0.9300,1.0230,1.1160,1.2090,NaN
2,Albania,2523 90 00,Grey hydraulic cements,0.8600,0.0300,0.8900,0.9790,1.0680,1.1570,(A)


In [4]:
# Two columns are expected to have nulls:
# - indirect_emissions: electricity and hydrogen have no indirect component under CBAM methodology
# - production_route: only populated for iron/steel rows where the xlsx specifies a route
null_counts = defaults.isnull().sum()
null_pct = (null_counts / len(defaults) * 100).round(1)
print(pd.DataFrame({"null_count": null_counts, "null_pct": null_pct}).to_string())

                      null_count  null_pct
country                        0    0.0000
cn_code                        0    0.0000
description                    0    0.0000
direct_emissions               0    0.0000
indirect_emissions          7926   74.3000
total_emissions                0    0.0000
default_2026                   1    0.0000
default_2027                   0    0.0000
default_2028_onwards           0    0.0000
production_route            1033    9.7000


In [ ]:
# Each country should have the same set of CN codes, so row counts should be roughly equal.
# Outliers may indicate missing rows for a country or genuine scope differences in the source xlsx.
countries = sorted(defaults["country"].unique())
print("Distinct countries:", len(countries))

rows_per_country = defaults["country"].value_counts()
print("\nRow count distribution across countries:")
print(rows_per_country.describe())

print("\nCountries with fewer rows than the median:")
print(rows_per_country[rows_per_country < rows_per_country.median()])

Distinct countries: 119

Row count distribution across countries:
count   119.0000
mean     89.6723
std      97.7892
min       1.0000
25%      28.0000
50%      51.0000
75%     220.5000
max     259.0000
Name: count, dtype: float64

Countries with fewer rows than the median (possible missing CN codes):
country
United Arab Emirates               34
Dominican Republic                 33
Georgia                            32
Lebanon                            32
Belarus                            31
Cuba                               31
Iraq                               31
Libya                              31
North Korea                        31
Trinidad and Tobago                31
Turkmenistan                       31
Albania                            30
Kyrgyzstan                         30
Madagascar                         30
Mali                               30
Oman                               30
Singapore                          30
Sudan                              30
Yemen 

In [6]:
# CN codes are stored as spaced strings (e.g. "2523 10 00").
# Codes appear at 4, 6, and 8-digit lengths, all valid under the CBAM regulation.
# Stripping spaces is needed before any join with trade flow data (which stores CN codes as integers).
defaults["cn_stripped"] = defaults["cn_code"].str.replace(" ", "")
defaults["cn_len"] = defaults["cn_stripped"].str.len()

print("CN code length distribution (after stripping spaces):")
print(defaults["cn_len"].value_counts().sort_index())

print("\nExample of each length:")
for length in sorted(defaults["cn_len"].unique()):
    sample = defaults[defaults["cn_len"] == length]["cn_code"].iloc[0]
    print(f"  {length}-digit: {sample!r}")

CN code length distribution (after stripping spaces):
cn_len
4    1111
6    1960
8    7600
Name: count, dtype: int64

Example of each length:
  4-digit: '7601'
  6-digit: '761090'
  8-digit: '2523 10 00'


In [7]:
# Descriptive stats across all emission columns.
# Key check: total_emissions should equal direct + indirect for all rows where indirect is not null.
# Any mismatch suggests a parsing error in extraction.
emission_cols = ["direct_emissions", "indirect_emissions", "total_emissions",
                 "default_2026", "default_2027", "default_2028_onwards"]
print("Emission value ranges (tCO2/t):")
print(defaults[emission_cols].describe().to_string())

print("\nZero and negative values (unexpected in emission defaults):")
for col in ["direct_emissions", "total_emissions"]:
    zeros = (defaults[col] == 0).sum()
    negs  = (defaults[col] < 0).sum()
    print(f"  {col}: {zeros} zeros, {negs} negatives")

check = defaults.dropna(subset=["indirect_emissions"]).copy()
check["diff"] = (check["total_emissions"] - (check["direct_emissions"] + check["indirect_emissions"])).abs()
mismatches = check[check["diff"] > 0.01]
print(f"\nRows where total != direct + indirect (tolerance 0.01): {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches[["country", "cn_code", "direct_emissions",
                       "indirect_emissions", "total_emissions", "diff"]].to_string())

Emission value ranges (tCO2/t):
       direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards
count        10671.0000           2745.0000       10671.0000    10670.0000    10671.0000            10671.0000
mean             2.5650              0.0743           2.5840        2.8150        3.0429                3.2709
std              1.8128              0.0382           1.8004        1.9938        2.1901                2.3880
min              0.0000              0.0000           0.0000        0.0000        0.0000                0.0000
25%              1.3900              0.0500           1.4400        1.5290        1.5960                1.6770
50%              2.3130              0.0700           2.3300        2.5410        2.7720                3.0030
75%              3.2100              0.1000           3.2100        3.5310        3.8520                4.1730
max             26.6400              0.3500          26.6400       29.3040      

In [8]:
# The CBAM regulation applies a fixed markup to total_emissions to derive default values:
# 10% for 2026, 20% for 2027, 30% for 2028 onwards, for most sectors.
# Fertilizers use a 1% markup. Checking the actual multipliers confirms extraction was clean
# and identifies any sectors with different markup schedules.
sample = defaults.dropna(subset=["default_2026"]).copy()
sample["mult_2026"] = (sample["default_2026"] / sample["total_emissions"]).round(4)
sample["mult_2027"] = (sample["default_2027"] / sample["total_emissions"]).round(4)
sample["mult_2028"] = (sample["default_2028_onwards"] / sample["total_emissions"]).round(4)

print("Uplift multipliers by sector (distinct combinations):")
print(sample.groupby(["mult_2026", "mult_2027", "mult_2028"]).size()
      .reset_index(name="row_count").sort_values("row_count", ascending=False).to_string(index=False))

Uplift multipliers by sector (distinct combinations):
 mult_2026  mult_2027  mult_2028  row_count
    1.1000     1.2000     1.3000       8275
    1.0100     1.0100     1.0100       2389
    1.1000     1.2100     1.3310          5


In [9]:
# production_route is only populated for iron/steel rows.
# Values correspond to production route codes defined in the CBAM regulation annex.
# Null = sector does not use route-based benchmarking (cement, fertilizers, hydrogen, electricity).
print("Production route values (null = non-steel sector):")
print(defaults["production_route"].value_counts(dropna=False).to_string())

Production route values (null = non-steel sector):
production_route
           3452
(C)        2675
NaN        1033
(F)         884
(L)         864
(K)         720
(E)         515
(A)         171
(H)         170
(C)/(F)     130
(B)          32
(E)/(H)      25


### 1. Observations

**Nulls**
- `indirect_emissions` is null for 74.3% of rows (7,926). Expected: electricity and
  hydrogen CN codes carry no indirect component under CBAM methodology.
- `production_route` is null for 9.7% of rows (1,033). Expected: only iron/steel rows
  carry a route code. Non-null values are letter codes (A, B, C, etc.) referencing
  production route definitions in the CBAM regulation annex.
- `default_2026` has 1 null. Needs investigation in cleaning — likely a row where the
  source xlsx held a dash or placeholder rather than a numeric value.

**Country coverage**
- 119 countries. Row counts vary significantly (min 1, max 259, median 51). The high
  variance is expected: larger industrial exporters have more CN codes covered. However,
  countries with very low row counts (Angola 9, Congo 4, Jamaica 4, several at 1-3)
  should be verified — they may reflect genuine scope limits in the source xlsx or
  extraction gaps.
- `Democratic Republic of the Cong` is a truncated country name, almost certainly
  "Democratic Republic of the Congo". Flag for standardization in cleaning.
- `Myanmar_Burma` uses an underscore separator rather than a slash or "and". Standardize
  in cleaning.
- `Côte d'Ivoire` and `Curaçao` contain non-ASCII characters. Confirm these survive
  encoding round-trips correctly.

**CN code format**
- Stored as spaced strings (e.g. `"2523 10 00"`). Three lengths present after stripping
  spaces: 4-digit (1,111 rows), 6-digit (1,960 rows), 8-digit (7,600 rows). All valid
  under the CN nomenclature. Spaces must be stripped before any join with trade flow data,
  which stores CN codes as integers.

**Emission values**
- 533 rows where `total_emissions != direct + indirect` at a 0.01 tolerance. All
  differences are exactly 0.01, spread across many countries and sectors. This is a
  rounding artifact from the source xlsx, not an extraction error — the values were
  published at 2 decimal places and the sum of two 2dp values occasionally rounds
  differently than the stored total. Not a data quality issue, but worth documenting.
  No action needed in cleaning.
- 1 row where `direct_emissions` and `total_emissions` are both 0. Investigate in
  cleaning.

**Uplift multipliers**
- Three distinct markup schedules confirmed:
  - 10/20/30% (×1.10/1.20/1.30): 8,275 rows — standard sectors (cement, steel, aluminium)
  - 1/1/1% (×1.01/1.01/1.01): 2,389 rows — fertilizers
  - 10/21/33.1% (×1.10/1.21/1.331): 5 rows — compounded markup, investigate which
    CN codes these are
- Extraction is clean. Multipliers are consistent with the regulation.